In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats

## Time Series Classification: Window Splitting
Window size and step size can be changed below. To keep as much data as possible, we have selected a step size of 1. To create window of ~25 seconds, we've selected a window size of 5.

In [ ]:
#window size, approx. 25 seconds
window_size = 5

#step size
step_size = 1

#participant ID 
pid = "009"

# task condition to be windowed
condition = "RP"

In [ ]:
participant_df = pd.read_csv("{}_{}_LabeledData.csv".format(pid, condition))

labels = ['continuous_optimal', 'binary_optimal', 'discrete_optimal']

#Features being used
all_neural_features = ['L_O_DSphi', 'L_D_DSphi', 'R_D_DSphi', 'R_O_DSphi', 'L_O_DSI', 'R_D_DSI', 'R_O_DSI', 'L_D_DSI']
phasic_neural_features = ['L_O_DSphi', 'L_D_DSphi', 'R_D_DSphi', 'R_O_DSphi']
intensity_neural_features = ['L_O_DSI', 'R_D_DSI', 'R_O_DSI', 'L_D_DSI']

In [76]:
def create_window_df(df, window_size:int, step_size:int, features, labels):
    data = {}

    # Add features
    for feature in features:
        data[feature] = df[feature]
    
    # Add labels
    for label in labels:
        data[label] = df[label]

    df = pd.DataFrame(data)

    windowed_data = []
    windowed_labels = []
    slope_values = []
    intercept_values = []

    for start in range(0, len(df) - window_size + 1, step_size):
        data_dict = {}
        end = start + window_size
        window = df.iloc[start:end] 

        last_discrete_label = window["discrete_optimal"].iloc[-1]
        last_continuous_label = window['continuous_optimal'].iloc[-1]
        last_binary_label = window['binary_optimal'].iloc[-1]

        window = window.drop(columns=['continuous_optimal', 'binary_optimal', 'discrete_optimal'])

        # Calculate window features
        mean_values = window.mean(axis=0).to_numpy(dtype=float)
        std_values = window.std(axis=0).to_numpy(dtype=float)
        slope_values = np.array([np.polyfit(window[feature], np.arange(window_size), 1)[0] for feature in features], dtype=float)
        intercept_values = np.array([np.polyfit(window[feature], np.arange(window_size), 1)[1] for feature in features], dtype=float)
        kurtosis_values = stats.kurtosis(window, axis=0, fisher=True)
        skewdness_values = stats.skew(window, axis=0)

        #Add feaures
        for i, feature in enumerate(features):
            data_dict[feature] = np.array([mean_values[i], std_values[i], slope_values[i], intercept_values[i], kurtosis_values[i], skewdness_values[i]])

        windowed_data.append(data_dict)
        windowed_labels.append({'discrete_label': last_discrete_label, 'continuous_label': last_continuous_label, 'binary_label': last_binary_label})

    windowed_data = pd.DataFrame(windowed_data)
    windowed_labels_df = pd.DataFrame(windowed_labels)

    return windowed_data, windowed_labels_df

In [ ]:
window_data, window_labels = create_window_df(participant_df, window_size=window_size, step_size=1, labels=labels, features=all_neural_features)

#concatenate labels and windowed data
df_concat_rows = pd.concat([window_labels, window_data], axis=1)

df_concat_rows["pid"] = pid
df_concat_rows["condition"] = condition


In [ ]:
df_concat_rows.tail(2)

,discrete_label,continuous_label,binary_label,L_O_DSphi,L_D_DSphi,R_D_DSphi,R_O_DSphi,L_O_DSI,R_D_DSI,R_O_DSI,L_D_DSI,pid,condition
2147,2,0.316004,1,"[5.9010596115463505, 0.5320969949414568, -2.96...","[0.3770621864396542, 0.19783276107022016, 7.98...","[1.8035235645604868, 0.042138803659858746, 37....","[-11.5971697955862, 0.02440384869558415, 64.06...","[2.6250030202117522, 0.051631724147430814, -30...","[-0.36412391903416885, 0.0032188095224243663, ...","[0.17487102869948656, 0.010123931666266957, 15...","[-0.7170494773266279, 0.016964831803540498, 91...",009,RP
2148,2,0.316004,1,"[5.588577912474254, 0.4761978923753183, -3.307...","[0.4963870453717448, 0.18561610341139104, 8.50...","[1.826917111886327, 0.03247781983328129, 47.93...","[-11.58749025029847, 0.016817234759661192, 77....","[2.588042285628762, 0.06691609431695146, -23.4...","[-0.36496824837602876, 0.0016368962791337026, ...","[0.18231448242406162, 0.012507940867143451, 12...","[-0.7051783286710814, 0.021325251167376994, 73...",009,RP


In [79]:
# Save as csv file
df_concat_rows.to_csv("{}_{}_{}Windows".format(pid, condition, window_size), index=True)